In [1]:
# =========================================================
# 01. INSTALACIÓN Y CONFIGURACIÓN DE PYSPARK
# =========================================================

!pip install pyspark -q

In [2]:
# =========================================================
# 02. INICIALIZACIÓN DE SPARK
# =========================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataSaludPeru") \
    .getOrCreate()

print("Spark iniciado correctamente")

Spark iniciado correctamente


In [3]:
# =========================================================
# 03. CARGA DEL DATASET DE ATENCIONES SIS
# =========================================================

from google.colab import files

uploaded = files.upload()

Saving 02_Muestra_Atenciones_SIS.xlsx to 02_Muestra_Atenciones_SIS.xlsx


In [5]:
# =========================================================
# 04. LECTURA DEL DATASET CON PANDAS
# =========================================================

!pip install xlrd -q

import pandas as pd

df_pandas = pd.read_excel("02_Muestra_Atenciones_SIS.xlsx")
print("Cantidad de registros:", len(df_pandas))
print("Cantidad de columnas:", len(df_pandas.columns))

df_pandas.head()

Cantidad de registros: 140
Cantidad de columnas: 17


,AÑO,MES,REGION,PROVINCIA,UBIGEO_DISTRITO,DISTRITO,COD_UNIDAD_EJECUTORA,DESC_UNIDAD_EJECUTORA,COD_IPRESS,IPRESS,NIVEL_EESS,PLAN_DE_SEGURO,COD_SERVICIO,DESC_SERVICIO,SEXO,GRUPO_EDAD,ATENCIONES
0,2020,9,LA LIBERTAD,PATAZ,130808,PARCOY,845,REGION LA LIBERTAD - SALUD,5387,VAQUERIA DE ANDAS,I,SIS GRATUITO,62,ATENCIÓN POR EMERGENCIA,FEMENINO,18 - 29 AÑOS,1
1,2020,9,LA LIBERTAD,PATAZ,130808,PARCOY,845,REGION LA LIBERTAD - SALUD,5387,VAQUERIA DE ANDAS,I,SIS PARA TODOS,9,ATENCIÓN PRENATAL,FEMENINO,30 - 59 AÑOS,1
2,2020,9,LA LIBERTAD,PATAZ,130808,PARCOY,845,REGION LA LIBERTAD - SALUD,16974,UCHUCUAYO,I,SIS GRATUITO,9,ATENCIÓN PRENATAL,FEMENINO,30 - 59 AÑOS,1
3,2020,9,LA LIBERTAD,PATAZ,130808,PARCOY,845,REGION LA LIBERTAD - SALUD,16974,UCHUCUAYO,I,SIS GRATUITO,56,CONSULTA EXTERNA,FEMENINO,00 - 04 AÑOS,5
4,2020,9,LA LIBERTAD,PATAZ,130808,PARCOY,845,REGION LA LIBERTAD - SALUD,16974,UCHUCUAYO,I,SIS GRATUITO,56,CONSULTA EXTERNA,MASCULINO,60 - MAS AÑOS,7


In [6]:
# =========================================================
# 05. CREACIÓN DEL DATAFRAME DE SPARK
# =========================================================

df_spark = spark.createDataFrame(df_pandas)

print("Registros en Spark:", df_spark.count())
print("Columnas en Spark:", len(df_spark.columns))

df_spark.show(5, truncate=False)

Registros en Spark: 140
Columnas en Spark: 17
+----+---+-----------+---------+---------------+--------+--------------------+--------------------------+----------+-----------------+----------+--------------+------------+-----------------------+---------+-------------+----------+
|AÑO |MES|REGION     |PROVINCIA|UBIGEO_DISTRITO|DISTRITO|COD_UNIDAD_EJECUTORA|DESC_UNIDAD_EJECUTORA     |COD_IPRESS|IPRESS           |NIVEL_EESS|PLAN_DE_SEGURO|COD_SERVICIO|DESC_SERVICIO          |SEXO     |GRUPO_EDAD   |ATENCIONES|
+----+---+-----------+---------+---------------+--------+--------------------+--------------------------+----------+-----------------+----------+--------------+------------+-----------------------+---------+-------------+----------+
|2020|9  |LA LIBERTAD|PATAZ    |130808         |PARCOY  |845                 |REGION LA LIBERTAD - SALUD|5387      |VAQUERIA DE ANDAS|I         |SIS GRATUITO  |62          |ATENCIÓN POR EMERGENCIA|FEMENINO |18 - 29 AÑOS |1         |
|2020|9  |LA LIBERTAD|

In [7]:
# =========================================================
# 06. VISUALIZACIÓN DEL ESQUEMA DEL DATAFRAME
# =========================================================

df_spark.printSchema()

root
 |-- AÑO: long (nullable = true)
 |-- MES: long (nullable = true)
 |-- REGION: string (nullable = true)
 |-- PROVINCIA: string (nullable = true)
 |-- UBIGEO_DISTRITO: long (nullable = true)
 |-- DISTRITO: string (nullable = true)
 |-- COD_UNIDAD_EJECUTORA: long (nullable = true)
 |-- DESC_UNIDAD_EJECUTORA: string (nullable = true)
 |-- COD_IPRESS: long (nullable = true)
 |-- IPRESS: string (nullable = true)
 |-- NIVEL_EESS: string (nullable = true)
 |-- PLAN_DE_SEGURO: string (nullable = true)
 |-- COD_SERVICIO: long (nullable = true)
 |-- DESC_SERVICIO: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- GRUPO_EDAD: string (nullable = true)
 |-- ATENCIONES: long (nullable = true)



In [8]:
# =========================================================
# 07. CREACIÓN DEL RDD DE ATENCIONES
# =========================================================

rdd_atenciones = df_spark.rdd.map(
    lambda fila: (
        fila["REGION"],
        fila["ATENCIONES"]
    )
)

print("Cantidad de registros en el RDD:", rdd_atenciones.count())

rdd_atenciones.take(5)

Cantidad de registros en el RDD: 140


[('LA LIBERTAD', 1),
 ('LA LIBERTAD', 1),
 ('LA LIBERTAD', 1),
 ('LA LIBERTAD', 5),
 ('LA LIBERTAD', 7)]

In [9]:
# =========================================================
# 08. AGREGACIÓN DE ATENCIONES POR REGIÓN CON RDD
# =========================================================

rdd_por_region = (
    rdd_atenciones
    .mapValues(lambda valor: int(valor))
    .reduceByKey(lambda a, b: a + b)
)

print("Atenciones acumuladas por región:")

for region, total in rdd_por_region.collect():
    print(region, "->", total)

Atenciones acumuladas por región:
LA LIBERTAD -> 1098


In [10]:
# =========================================================
# 09. CREACIÓN DE VISTA TEMPORAL PARA SPARKSQL
# =========================================================

df_spark.createOrReplaceTempView("atenciones_sis")

print("Vista temporal 'atenciones_sis' creada correctamente")

Vista temporal 'atenciones_sis' creada correctamente


In [11]:
# =========================================================
# 10. CONSULTA MEDIANTE SPARKSQL
# =========================================================

resultado_sql = spark.sql("""
    SELECT
        REGION,
        SUM(ATENCIONES) AS TOTAL_ATENCIONES
    FROM atenciones_sis
    GROUP BY REGION
    ORDER BY TOTAL_ATENCIONES DESC
""")

resultado_sql.show(truncate=False)

+-----------+----------------+
|REGION     |TOTAL_ATENCIONES|
+-----------+----------------+
|LA LIBERTAD|1098            |
+-----------+----------------+



In [12]:
# =========================================================
# 11. MEDICIÓN DEL TIEMPO DE EJECUCIÓN EN SPARK
# =========================================================

import time

inicio_spark = time.perf_counter()

resultado_sql = spark.sql("""
    SELECT
        REGION,
        SUM(ATENCIONES) AS TOTAL_ATENCIONES
    FROM atenciones_sis
    GROUP BY REGION
""")

resultado_sql.collect()

fin_spark = time.perf_counter()

tiempo_spark = fin_spark - inicio_spark

print(f"Tiempo de ejecución Spark: {tiempo_spark:.6f} segundos")

Tiempo de ejecución Spark: 0.787202 segundos


In [13]:
print(f"Tiempo de ejecución Spark: {tiempo_spark:.6f} segundos")

Tiempo de ejecución Spark: 0.787202 segundos
